# Learn BayMorph

This notebook runs the official fast end-to-end BayMorph tutorial on a controlled synthetic galaxy. It demonstrates the supported configuration-driven workflow:

1. load and inspect the official YAML configuration;
2. generate a reproducible mock observation;
3. prepare the image and noise model;
4. fit a single Sérsic model with Dynesty;
5. inspect posterior summaries and diagnostic plots;
6. compare the recovered parameters with the injected truth.

The exercise is intentionally small enough for a tutorial, but it uses the same public workflow components intended for scientific analyses.


## Scientific scope

The mock source is a single elliptical Sérsic profile convolved with a Gaussian PSF and observed with Gaussian read noise. The fit allows flux, effective radius, Sérsic index, axis ratio, position angle, centroid, and residual sky to vary.

A good fit does **not** require the injected truth to fall inside every marginal 68% credible interval for one noisy realization. In particular, total flux, effective radius, Sérsic index, and residual sky can be strongly correlated.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import yaml

from baymorph import BayMorphRunner, load_resolved_config
from baymorph.pipeline import run_simulation_from_config


def find_repository_root(start: Path | None = None) -> Path:
    """Return the nearest parent directory containing pyproject.toml."""
    current = (start or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate

    raise FileNotFoundError(
        "Run this notebook from the BayMorph repository or one of its subdirectories."
    )


REPOSITORY_ROOT = find_repository_root()
os.chdir(REPOSITORY_ROOT)

CONFIG_PATH = REPOSITORY_ROOT / "configs" / "learn_baymorph.yaml"
RUN_DIRECTORY = REPOSITORY_ROOT / "runs" / "learn_baymorph"

print("Repository:", REPOSITORY_ROOT)
print("Configuration:", CONFIG_PATH)


## 1. Inspect the official configuration

The tutorial configuration is the scientific record of the run. It specifies the injected source, image preparation, likelihood, priors, sampler, and diagnostics.


In [ ]:
with CONFIG_PATH.open() as stream:
    raw_config = yaml.safe_load(stream)

raw_config


In [ ]:
config = load_resolved_config(CONFIG_PATH)

print("Object:", config["object"]["name"])
print("Image shape:", config["simulation"]["shape"])
print("Model:", config["fit"]["models"])
print("Sampler:", config["fit"]["sampler"]["name"])
print("Live points:", config["fit"]["sampler"]["nlive"])
print("Diagnostics:", config["diagnostics"])


## 2. Generate the synthetic observation

The random seed in the YAML makes the realization reproducible. The saved FITS image includes the convolved source, sky level, and noise. BayMorph then estimates and subtracts the background during image preparation.


In [ ]:
observation = run_simulation_from_config(CONFIG_PATH)

print("Saved input image:", config["data"]["image_path"])
print("Image shape:", observation.data.shape)
print("PSF shape:", observation.psf_kernel.shape)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(observation.intrinsic_image, origin="lower")
axes[0].set_title("Intrinsic Sérsic model")

axes[1].imshow(observation.convolved_image, origin="lower")
axes[1].set_title("PSF-convolved model")

axes[2].imshow(observation.data, origin="lower")
axes[2].set_title("Final noisy observation")

for axis in axes:
    axis.set_xlabel("x [pixel]")
    axis.set_ylabel("y [pixel]")

fig.tight_layout()


## 3. Run the complete BayMorph workflow

`BayMorphRunner` is the supported high-level interface. For this Gaussian-PSF configuration, `run()` performs image preparation and Bayesian fitting. With `diagnostics=True`, it also generates corner, residual, and radial-profile diagnostics.


In [ ]:
runner = BayMorphRunner(CONFIG_PATH)
fit = runner.run(diagnostics=True)

print("Run directory:", RUN_DIRECTORY)
print("Model:", fit.model_name)
print("Reduced chi-square:", fit.reduced_chi2)


## 4. Inspect image preparation

The fitter uses the prepared image, empirical noise map, contaminant mask, and configured PSF. These are the actual products entering the likelihood.


In [ ]:
prepared = runner.image

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(prepared.prepared_data, origin="lower")
axes[0].set_title("Prepared image")

axes[1].imshow(prepared.noise, origin="lower")
axes[1].set_title("Noise map")

axes[2].imshow(prepared.contaminant_mask, origin="lower")
axes[2].set_title("Contaminant mask")

for axis in axes:
    axis.set_xlabel("x [pixel]")
    axis.set_ylabel("y [pixel]")

fig.tight_layout()

print("Estimated target centre:", prepared.target_x, prepared.target_y)
print("Masked pixels:", int(np.count_nonzero(prepared.contaminant_mask)))
print("Median noise:", float(np.median(prepared.noise)))


## 5. Compare injected and recovered parameters

BayMorph reports posterior summaries through the fit result. The best-parameter dictionary used below is the workflow's representative fitted solution. Posterior samples remain available for uncertainty and covariance analysis.


In [ ]:
truth = {
    "flux": float(config["simulation"]["truth"]["flux"]),
    "re": float(config["simulation"]["truth"]["re"]),
    "n": float(config["simulation"]["truth"]["n"]),
    "q": float(config["simulation"]["truth"]["q"]),
    "theta": np.deg2rad(float(config["simulation"]["truth"]["theta_deg"])),
    "x0": float(config["simulation"]["truth"]["x0"]),
    "y0": float(config["simulation"]["truth"]["y0"]),
    "sky": 0.0,
}

parameter_order = ("flux", "re", "n", "q", "theta", "x0", "y0", "sky")

print(f"{'parameter':<12}{'truth':>14}{'fit':>14}{'difference':>14}")
print("-" * 54)
for name in parameter_order:
    fitted = float(fit.best_parameters[name])
    injected = truth[name]
    print(f"{name:<12}{injected:>14.6f}{fitted:>14.6f}{fitted - injected:>14.6f}")

print()
print("Injected theta [deg]:", np.rad2deg(truth["theta"]))
print("Recovered theta [deg]:", np.rad2deg(fit.best_parameters["theta"]))


## 6. Inspect the posterior and residuals

The corner plot uses equal-weight posterior samples. Correlated contours are expected for the Sérsic `flux`–`re`–`n` combination and should be interpreted jointly rather than parameter by parameter.


In [ ]:
figure, _, _ = fit.corner()
figure.suptitle("Single-Sérsic posterior", y=1.02)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(prepared.prepared_data, origin="lower")
axes[0].set_title("Prepared data")

axes[1].imshow(fit.best_model_image, origin="lower")
axes[1].set_title("Best-fit model")

limit = np.nanpercentile(np.abs(fit.norm_residual), 99)
image = axes[2].imshow(
    fit.norm_residual,
    origin="lower",
    vmin=-limit,
    vmax=limit,
)
axes[2].set_title("Normalized residual")
fig.colorbar(image, ax=axes[2], label="Residual / noise")

for axis in axes:
    axis.set_xlabel("x [pixel]")
    axis.set_ylabel("y [pixel]")

fig.tight_layout()


In [ ]:
figure, _ = fit.radial_profile()
figure.suptitle("Radial-profile diagnostic", y=1.02)
plt.show()


## 7. Interpretation

For this realization, an acceptable reduced chi-square and structureless normalized residuals establish that the image model describes the data at the adopted noise level. Small shifts in individual Sérsic parameters do not by themselves indicate a simulation–fitting inconsistency.

The scientifically relevant checks are:

- the source centroid and geometry are recovered accurately;
- residuals are consistent with the noise model;
- the posterior exposes the covariance among flux, effective radius, Sérsic index, and sky;
- the injected parameters remain a reference point, not a requirement that every marginal 68% interval contain the truth in one realization.

The generated files under `runs/learn_baymorph/` provide a reproducible record of the configuration, prepared products, fit result, and diagnostics.
